# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's view all available RecordSets and their fields. All Croissant entities are referenced by their `@id` field as per the Croissant specification.

In [ ]:
# List all RecordSets by @id and their fields
record_sets = list(dataset.metadata.record_sets)
print(f"Number of RecordSets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}")
    print(f"  @id        : {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

For demonstration, we'll extract the first available record set, but you can select by `@id` from the overview above.

In [ ]:
# Fetch records for all RecordSets and load into DataFrames
dataframes = {}
record_sets = list(dataset.metadata.record_sets)
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    # Using @id to specify which RecordSet
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# For this notebook, let's just explore the first available RecordSet
if record_set_ids:
    first_recordset_id = record_set_ids[0]
    print(f"\nColumns in '{first_recordset_id}':")
    print(dataframes[first_recordset_id].columns.tolist())
    display(dataframes[first_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's select a numeric field and group/categorize by a categorical field. Both fields are referenced by their `@id`.

In [ ]:
# --- EDA Section: Filter, normalize, group ---
# Identify potential numeric and categorical field candidates

if record_set_ids:
    record_set_id = first_recordset_id
    df = dataframes[record_set_id]
    
    # Find numeric and non-numeric fields by @id
    record_set_obj = [rs for rs in dataset.metadata.record_sets if rs.id == record_set_id][0]
    numeric_field_id = None
    group_field_id = None

    for field in record_set_obj.fields:
        if field.data_type in ('Integer', 'Float', 'Number') and numeric_field_id is None:
            numeric_field_id = field.id
        if field.data_type == 'Text' and group_field_id is None:
            group_field_id = field.id

    # Fall back to first numeric/categorical column name if no explicit match
    if numeric_field_id is None:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
    else:
        # If the column with the field_id name doesn't exist, try its local part
        if numeric_field_id not in df.columns and ':' in numeric_field_id:
            numeric_field_id_simple = numeric_field_id.split(':', 1)[-1]
            if numeric_field_id_simple in df.columns:
                numeric_field_id = numeric_field_id_simple

    if group_field_id is None:
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
    else:
        if group_field_id not in df.columns and ':' in group_field_id:
            group_field_id_simple = group_field_id.split(':', 1)[-1]
            if group_field_id_simple in df.columns:
                group_field_id = group_field_id_simple

    print(f"Numeric field (@id): {numeric_field_id}")
    print(f"Group field (@id): {group_field_id}\n")

    # Filtering
    threshold = None
    if numeric_field_id and numeric_field_id in df.columns:
        # Compute a threshold as, say, the 75th percentile of values
        valid_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = valid_values.quantile(0.75)
        filtered_df = df[valid_values > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        if not filtered_df.empty:
            mean_val = filtered_df[numeric_field_id].astype(float).mean()
            std_val = filtered_df[numeric_field_id].astype(float).std()
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id].astype(float) - mean_val
            ) / std_val
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Group by
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
                display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's plot the distribution of the selected numeric field and the group means if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'grouped_df' in locals():
        plt.figure(figsize=(8, 4))
        top_n = grouped_df.sort_values(numeric_field_id, ascending=False).head(20)
        sns.barplot(y=group_field_id, x=numeric_field_id, data=top_n)
        plt.title(f"Top 20 groups by mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* In this notebook, we demonstrated how to use the `mlcroissant` library to programmatically load and analyze a FAIR ^2 mass spectrometry dataset defined by a Croissant schema.
* Entities in the dataset, including record sets and fields, are referenced via their `@id` for reproducibility.
* You can further extend this notebook by selecting different record sets, performing different EDA steps, or integrating with downstream machine learning workflows.

**See the Croissant schema and documentation for more advanced usage.**